# 01 — Chuẩn bị backup trên Colab

    ZIP gốc và checksum đã được upload vào `MyDrive/tram-can/`. Notebook copy ZIP về `/content`, kiểm tra SHA-256, giải nén rồi tạo index/split/pilot. **Gán nhãn bằng công cụ local trong README**; notebook này chưa tạo ground truth. Khi mount Drive, chọn tài khoản chứa thư mục `tram-can`.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
from google.colab import drive

REPO_URL = 'https://github.com/huybitvvt/scale-ocr.git'
PROJECT_COMMIT = ''  # Điền commit SHA cố định trước run chính; để trống chỉ khi thăm dò.
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'  # Tạo token đọc repo trong Colab Secrets.
VERSION = 'v001'
DRIVE_ROOT = Path('/content/drive/MyDrive/tram-can')
LOCAL_ROOT = Path('/content/scale-data')
REPO_ROOT = Path('/content/scale-ocr-code')
drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), 'Không thấy MyDrive/tram-can: mount đúng tài khoản Google chứa ZIP'
assert REPO_URL.startswith(('https://', 'git@')), 'Điền REPO_URL sau khi push Git'
if not REPO_ROOT.exists():
    git_env = os.environ.copy()
    askpass = Path('/content/scale-ocr-askpass.sh')
    try:
        if GITHUB_SECRET_NAME:
            from google.colab import userdata
            assert REPO_URL.startswith('https://github.com/'), 'Secret này chỉ dùng với github.com'
            token = userdata.get(GITHUB_SECRET_NAME)
            assert token, 'Chưa có Colab Secret GITHUB_TOKEN hoặc chưa cấp quyền notebook đọc secret'
            git_env['SCALE_GIT_TOKEN'] = token
            git_env['GIT_TERMINAL_PROMPT'] = '0'
            git_env['GIT_ASKPASS'] = str(askpass)
            askpass.write_text(chr(10).join([
                '#!/bin/sh', 'case "$1" in',
                '  *Username*) printf "%s\n" "x-access-token" ;;',
                '  *) printf "%s\n" "$SCALE_GIT_TOKEN" ;;',
                'esac', ''
            ]), encoding='utf-8')
            askpass.chmod(0o700)
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)],
                       env=git_env, check=True)
    finally:
        git_env.pop('SCALE_GIT_TOKEN', None)
        askpass.unlink(missing_ok=True)
if PROJECT_COMMIT:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', PROJECT_COMMIT], check=True)
PROJECT_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT)], check=True)
print('Code commit:', PROJECT_COMMIT)
print('Disk GiB free:', round(shutil.disk_usage('/content').free / 2**30, 2))
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
source = DRIVE_ROOT / 'cloudinary_backup_2026-09-27.zip'
sha_file = source.with_name(source.name + '.sha256')
assert source.is_file() and sha_file.is_file(), 'Không thấy ZIP và .sha256 trong MyDrive/tram-can; kiểm tra tài khoản Drive đã mount'
expected = sha_file.read_text(encoding='ascii').split()[0].lower()
assert len(expected) == 64
assert shutil.disk_usage('/content').free > source.stat().st_size * 2, 'Thiếu dung lượng local'
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
local_zip = LOCAL_ROOT / source.name
if not local_zip.exists():
    shutil.copyfile(source, local_zip)
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
assert sha256(local_zip) == expected, 'Backup SHA-256 mismatch; copy lại ZIP'
raw_root = LOCAL_ROOT / 'cloudinary_backup_2026-09-27'
if not (raw_root / 'manifest.json').is_file():
    with zipfile.ZipFile(local_zip) as archive:
        for item in archive.infolist():
            target = (LOCAL_ROOT / item.filename).resolve()
            assert target.is_relative_to(LOCAL_ROOT.resolve()), 'Unsafe ZIP path'
        archive.extractall(LOCAL_ROOT)
assert (raw_root / 'manifest.json').is_file()
print('Raw archive verified:', expected)


In [ ]:
work = LOCAL_ROOT / 'work' / VERSION
work.mkdir(parents=True, exist_ok=True)
def run(*args):
    subprocess.run([sys.executable, '-m', 'scale_ocr', *map(str, args)], check=True)
run('index', '--raw-root', raw_root, '--out', work / 'index.csv')
run('split', '--index', work / 'index.csv', '--out', work / 'split.csv')
run('pilot', '--index', work / 'index.csv', '--split', work / 'split.csv',
    '--out', work / 'pilot.csv', '--count', 400)
print('Pilot:', work / 'pilot.csv')
print('Để gán nhãn local: xem README. Lưu annotations.jsonl và split.csv trên Drive.')
